In [1]:
# # UniLSTM Logon Agent
# 
# This script trains the insider threat agent specifically for the LOGON modality using a Unidirectional LSTM.
# It uses a strict 1:1 RandomUnderSampler to combat the extreme class imbalance and "lazy model" problem.
# 
# Data partition: uses the OFFICIAL Week 1 user split (week1/week1_official_split.py ->
# data/user_splits/{train,val,test}_users.npy), 60% train / 20% validation / 20% test.

import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from imblearn.under_sampling import RandomUnderSampler
from sklearn.preprocessing import StandardScaler
import argparse
import os
from pathlib import Path
import mlflow
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import recall_score, confusion_matrix
import time

# ============================================================================
# Official Week 1 user split (single source of truth)
# ============================================================================
# The 60/20/20 user-level split is produced by week1/week1_official_split.py
# and saved to data/user_splits/{train,val,test}_users.npy. It is the ONLY
# partition authority -- this notebook never re-splits users.


# === Determinism: seed all RNGs for reproducible results (as in week3/tier1_score_generation.ipynb) ===
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Use a seeded generator for DataLoader shuffle
g = torch.Generator()
g.manual_seed(SEED)

print(f"All RNGs seeded with SEED={SEED}")

def _find_project_root():
    """Walk up to pyproject.toml (project portability convention)."""
    _curr = Path.cwd()
    while not (_curr / 'pyproject.toml').exists() and _curr != _curr.parent:
        _curr = _curr.parent
    return _curr

def load_official_user_split(split_dir=None):
    """Load the official Week 1 user split from data/user_splits/.

    Returns (train_users, val_users, test_users) as numpy arrays of user IDs.
    Raises FileNotFoundError if the split files are missing.
    """
    if split_dir is None:
        split_dir = _find_project_root() / "data" / "user_splits"
    split_dir = Path(split_dir)
    files = {
        "train": split_dir / "train_users.npy",
        "val": split_dir / "val_users.npy",
        "test": split_dir / "test_users.npy",
    }
    for key, path in files.items():
        if not path.exists():
            raise FileNotFoundError(
                f"Official split file not found: {path}. "
                f"Run week1/week1_official_split.py first."
            )
    train_users = np.load(files["train"])
    val_users = np.load(files["val"])
    test_users = np.load(files["test"])

    # Explicit no-overlap assertions
    assert len(set(train_users) & set(val_users)) == 0, "train ∩ validation must be empty"
    assert len(set(train_users) & set(test_users)) == 0, "train ∩ test must be empty"
    assert len(set(val_users) & set(test_users)) == 0, "validation ∩ test must be empty"
    assert len(set(train_users) | set(val_users) | set(test_users)) == len(train_users) + len(val_users) + len(test_users), "all users must be accounted for"

    n = len(train_users) + len(val_users) + len(test_users)
    print("=" * 60)
    print("OFFICIAL USER SPLIT (from week1/week1_official_split.py)")
    print("=" * 60)
    print(f"Total users:       {n}")
    print(f"Training users:    {len(train_users)} ({100*len(train_users)/n:.0f}%)")
    print(f"Validation users:  {len(val_users)} ({100*len(val_users)/n:.0f}%)")
    print(f"Testing users:     {len(test_users)} ({100*len(test_users)/n:.0f}%)")
    print(f"Train ∩ Validation: {len(set(train_users) & set(val_users))}")
    print(f"Train ∩ Test:       {len(set(train_users) & set(test_users))}")
    print(f"Validation ∩ Test:  {len(set(val_users) & set(test_users))}\n")
    return train_users, val_users, test_users

class ModalityDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = torch.tensor(sequences, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

def get_modality_columns(all_cols, modality):
    cols = []
    personality = ['O', 'C', 'E', 'A', 'N']
    if modality == 'logon':
        cols = [c for c in all_cols if 'logon' in c or 'logout' in c]
    elif modality == 'email':
        cols = [c for c in all_cols if 'email' in c or 'mail' in c]
    elif modality == 'http':
        cols = [c for c in all_cols if 'http' in c or 'url' in c]
    elif modality == 'file_device':
        keywords = ['file', 'device', 'comp', 'disk', 'phof', 'docf', 'txtf', 'exef']
        cols = [c for c in all_cols if any(k in c for k in keywords)]
    else:
        raise ValueError(f"Unknown modality: {modality}")
        
    return list(set(cols)) + personality

def create_sequences_and_labels(df, feature_cols, seq_length=7):
    # Sort data
    df = df.sort_values(by=['user', 'week', 'day']).reset_index(drop=True)
    
    sequences = []
    labels = []
    
    # Group by user to prevent sequence overlap between different users
    for user, user_data in df.groupby('user'):
        features = user_data[feature_cols].values
        # Binarize insider label: 0 for normal, 1 for any scenario
        user_labels = (user_data['insider'].values > 0).astype(int)
        
        # Sliding window
        for i in range(len(features) - seq_length + 1):
            seq = features[i:i + seq_length]
            # Label is 1 if any day in the sequence has an insider event
            seq_label = user_labels[i:i + seq_length].max()
            
            sequences.append(seq)
            labels.append(seq_label)
            
    return np.array(sequences, dtype=np.float32), np.array(labels, dtype=np.int8)

def get_data_loaders(df, modality, split_dir=None, seq_length=7, batch_size=128):
    
    # Get feature columns for this modality
    feature_cols = get_modality_columns(df.columns.tolist(), modality)
    print(f"Selected {len(feature_cols)} features for {modality}.")
    
    # Use the OFFICIAL Week 1 user split (60/20/20) -- no random re-split here.
    train_users, val_users, test_users = load_official_user_split(split_dir=split_dir)
    
    train_df = df[df['user'].isin(train_users)].copy()
    val_df = df[df['user'].isin(val_users)].copy()
    test_df = df[df['user'].isin(test_users)].copy()
    
    # Verify user-level separation is airtight on the actual dataframe.
    train_set = set(train_df['user'].unique())
    val_set = set(val_df['user'].unique())
    test_set = set(test_df['user'].unique())
    assert not (train_set & val_set), "User leakage: train ∩ validation"
    assert not (train_set & test_set), "User leakage: train ∩ test"
    assert not (val_set & test_set), "User leakage: validation ∩ test"
    
    # Scale features (fit on train, transform on val/test to avoid leakage)
    scaler = StandardScaler()
    train_df[feature_cols] = train_df[feature_cols].astype(np.float32)
    val_df[feature_cols] = val_df[feature_cols].astype(np.float32)
    test_df[feature_cols] = test_df[feature_cols].astype(np.float32)
    train_df.loc[:, feature_cols] = scaler.fit_transform(train_df[feature_cols].fillna(0)).astype(np.float32)
    val_df.loc[:, feature_cols] = scaler.transform(val_df[feature_cols].fillna(0)).astype(np.float32)
    test_df.loc[:, feature_cols] = scaler.transform(test_df[feature_cols].fillna(0)).astype(np.float32)
    
    # Create sequences separately per partition (per-user, no cross-user windows)
    X_train, y_train = create_sequences_and_labels(train_df, feature_cols, seq_length)
    X_val, y_val = create_sequences_and_labels(val_df, feature_cols, seq_length)
    X_test, y_test = create_sequences_and_labels(test_df, feature_cols, seq_length)
    
    print("\n============================================================")
    print("DATASET SUMMARY")
    print("============================================================")
    print(f"{'Split':<12}{'Users':>8}{'Sequences':>12}{'Positive':>12}")
    print("-" * 44)
    print(f"{'Train':<12}{len(train_set):>8}{len(X_train):>12}{int(y_train.sum()):>12}")
    print(f"{'Validation':<12}{len(val_set):>8}{len(X_val):>12}{int(y_val.sum()):>12}")
    print(f"{'Test':<12}{len(test_set):>8}{len(X_test):>12}{int(y_test.sum()):>12}")
    print("-" * 44 + "\n")
    
    # Apply Random Undersampling to TRAIN SET ONLY (existing strategy: 1:1)
    # We must flatten the sequence to use imbalanced-learn
    n_samples, seq_len, n_features = X_train.shape
    X_train_flat = X_train.reshape(n_samples, -1)
    
    # NOTE: Strictly using 1:1 ratio (sampling_strategy=1.0)
    rus = RandomUnderSampler(sampling_strategy=1.0, random_state=42)
    X_train_resampled, y_train_resampled = rus.fit_resample(X_train_flat, y_train)
    
    # Unflatten back to sequences
    X_train_resampled = X_train_resampled.reshape(-1, seq_len, n_features)
    
    print(f"Resampled training sequences: {len(X_train_resampled)} | Positives: {y_train_resampled.sum()}")
    
    train_dataset = ModalityDataset(X_train_resampled, y_train_resampled)
    val_dataset = ModalityDataset(X_val, y_val)
    test_dataset = ModalityDataset(X_test, y_test)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, generator=g)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    return train_loader, val_loader, test_loader, n_features

class ModalityLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=1, dropout=0.2):
        super(ModalityLSTM, self).__init__()
        
        # LSTM for sequence modeling
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=False # Strictly Unidirectional
        )
        
        # Fully connected layer for binary classification (Normal vs Insider)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        # x shape: (batch_size, seq_length, input_dim)
        lstm_out, (h_n, c_n) = self.lstm(x)
        
        # Unidirectional: just take the final hidden state
        final_hidden = h_n[-1, :, :]
        
        # Output logit
        out = self.fc(final_hidden)
        return out.squeeze(1)

# Resolve project root
_curr = Path.cwd()
while not (_curr / 'pyproject.toml').exists() and _curr != _curr.parent:
    _curr = _curr.parent
PROJECT_ROOT = _curr
DATA_DIR = PROJECT_ROOT / "data"
MLFLOW_DB = PROJECT_ROOT / "mlflow.db"

def _evaluate_loader(model, loader, device):
    """Run the existing binary-evaluation loop on a DataLoader.

    Uses the exact same metrics as the original implementation: binary
    predictions at threshold 0.5, recall (TPR), FPR from the confusion
    matrix. Returns (recall, fpr, tn, fp, fn, tp, n_samples).
    """
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)

            # Apply sigmoid and threshold at 0.5
            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).int()

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

    recall = recall_score(all_labels, all_preds, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(all_labels, all_preds).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    return recall, fpr, int(tn), int(fp), int(fn), int(tp), len(all_labels)


def train(modality, df, epochs=20, lr=0.001, seq_length=7, batch_size=128, hidden_dim=64):
    # Robust device selection: probe CUDA with a real op, fall back to CPU.
    # Some GPUs (e.g. RTX 5090 / sm_120) report cuda.is_available()==True but
    # have no kernel image in the installed torch build, which crashes on first
    # use. This matches the _cuda_usable() convention in week2/experiments/.
    def _cuda_usable():
        if not torch.cuda.is_available():
            return False
        try:
            x = torch.randn(4, 4, device='cuda')
            _ = x @ x
            torch.cuda.synchronize()
            return True
        except Exception:
            return False

    device = torch.device('cuda' if _cuda_usable() else 'cpu')
    print(f"Training {modality} agent on device: {device}")
    
    # 1. Setup MLflow
    mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB}")
    mlflow.set_experiment("Week2_UniLSTM_Logon_Agent")
    
    with mlflow.start_run(run_name=f"{modality}_agent"):
        # Log Hyperparameters
        mlflow.log_params({
            "modality": modality,
            "epochs": epochs,
            "learning_rate": lr,
            "sequence_length": seq_length,
            "batch_size": batch_size,
            "hidden_dim": hidden_dim,
            "model_type": "Uni-LSTM",
            "balancing": "RandomUnderSampler (Normal:Anomaly 1:1)",
            "split_type": "official_60_20_20_user_split",
            "train_ratio": 0.60,
            "validation_ratio": 0.20,
            "test_ratio": 0.20
        })
        
        # 2. Get Data (official Week 1 split -> train / validation / test)
        train_loader, val_loader, test_loader, input_dim = get_data_loaders(
            df=df, 
            modality=modality, 
            seq_length=seq_length, 
            batch_size=batch_size
        )
        
        mlflow.log_param("input_dim", input_dim)
        
        # Log official-split user counts (actual counts from the official split)
        _tu, _vu, _teu = load_official_user_split()
        mlflow.log_params({
            "train_user_count": int(len(_tu)),
            "validation_user_count": int(len(_vu)),
            "test_user_count": int(len(_teu)),
            "random_seed": 42
        })
        
        if input_dim <= 5:
            print(f"Warning: Only {input_dim} features found for {modality}. Check column matching.")
            
        # 3. Setup Model
        model = ModalityLSTM(input_dim=input_dim, hidden_dim=hidden_dim).to(device)
        criterion = nn.BCEWithLogitsLoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)
        
        # 4. Training Loop
        for epoch in range(epochs):
            model.train()
            total_loss = 0
            
            for X_batch, y_batch in train_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                
                optimizer.zero_grad()
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                
                loss.backward()
                optimizer.step()
                
                total_loss += loss.item()
                
            avg_loss = total_loss / len(train_loader)
            if (epoch + 1) % 5 == 0 or epoch == 0:
                print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f}")
                mlflow.log_metric("train_loss", avg_loss, step=epoch)
                
        # 5. Evaluation (validation + test, both untouched during training)
        val_recall, val_fpr, val_tn, val_fp, val_fn, val_tp, val_n = _evaluate_loader(model, val_loader, device)
        test_recall, test_fpr, test_tn, test_fp, test_fn, test_tp, test_n = _evaluate_loader(model, test_loader, device)

        print(f"--- {modality.upper()} AGENT EVALUATION (VALIDATION, WINDOW SIZE: {seq_length}) ---")
        print(f"Recall (TPR): {val_recall:.4f}")
        print(f"False Positive Rate (FPR): {val_fpr:.4f}")
        print(f"Confusion Matrix:\nTN={val_tn} FP={val_fp}\nFN={val_fn} TP={val_tp}")

        print(f"--- {modality.upper()} AGENT EVALUATION (TEST, WINDOW SIZE: {seq_length}) ---")
        print(f"Recall (TPR): {test_recall:.4f}")
        print(f"False Positive Rate (FPR): {test_fpr:.4f}")
        print(f"Confusion Matrix:\nTN={test_tn} FP={test_fp}\nFN={test_fn} TP={test_tp}")

        # 6. Structured comparison table (validation vs test)
        print("\n" + "=" * 60)
        print(f"{modality.upper()} — VALIDATION vs TEST (window {seq_length}, official 60/20/20 split)")
        print("=" * 60)
        print(f"{'Dataset':<12}{'Samples':>9}{'Recall':>9}{'FPR':>9}{'TN':>7}{'FP':>7}{'FN':>7}{'TP':>7}")
        print("-" * 60)
        print(f"{'Validation':<12}{val_n:>9}{val_recall:>9.4f}{val_fpr:>9.4f}{val_tn:>7}{val_fp:>7}{val_fn:>7}{val_tp:>7}")
        print(f"{'Test':<12}{test_n:>9}{test_recall:>9.4f}{test_fpr:>9.4f}{test_tn:>7}{test_fp:>7}{test_fn:>7}{test_tp:>7}")
        print("=" * 60 + "\n")

        # Log Metrics (both partitions)
        mlflow.log_metrics({
            "val_recall": val_recall,
            "val_fpr": val_fpr,
            "val_true_positives": val_tp,
            "val_false_positives": val_fp,
            "val_true_negatives": val_tn,
            "val_false_negatives": val_fn,
            "test_recall": test_recall,
            "test_fpr": test_fpr,
            "true_positives": int(test_tp),
            "false_positives": int(test_fp),
            "true_negatives": int(test_tn),
            "false_negatives": int(test_fn)
        })
        
        # Save Model
        save_dir = Path.cwd() / "saved_models"
        save_dir.mkdir(exist_ok=True)
        model_path = save_dir / f"{modality}_unilstm_agent.pt"
        torch.save(model.state_dict(), model_path)
        mlflow.log_artifact(str(model_path))
        
        print(f"Finished {modality}. Run logged to MLflow.\n")

if __name__ == "__main__":
    print('='*50)
    print('STARTING PIPELINE FOR LOGON AGENT (UniLSTM, 1:1 Ratio)')
    print('='*50)
    
    print("Loading dataset once to prevent memory errors...")
    df = pd.read_csv(str(DATA_DIR / "dayr4.2.csv"))
    # Downcast memory to prevent MemoryError
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = df[col].astype('float32')
    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')
        
    for window in [3, 5, 7]:
        print(f"\n{'*'*40}")
        print(f"EVALUATING WINDOW SIZE: {window} DAYS")
        print(f"{'*'*40}")
        train('logon', df, epochs=20, seq_length=window)


All RNGs seeded with SEED=42
STARTING PIPELINE FOR LOGON AGENT (UniLSTM, 1:1 Ratio)
Loading dataset once to prevent memory errors...

****************************************
EVALUATING WINDOW SIZE: 3 DAYS
****************************************
Training logon agent on device: cpu


C:\insider_threat_detection_project2\.venv\Lib\site-packages\torch\cuda\__init__.py:235: UserWarning: 
NVIDIA GeForce RTX 5090 with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_61 sm_70 sm_75 sm_80 sm_86 sm_90.
If you want to use the NVIDIA GeForce RTX 5090 GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


Selected 20 features for logon.
OFFICIAL USER SPLIT (from week1/week1_official_split.py)
Total users:       1000
Training users:    600 (60%)
Validation users:  200 (20%)
Testing users:     200 (20%)
Train ∩ Validation: 0
Train ∩ Test:       0
Validation ∩ Test:  0


DATASET SUMMARY
Split          Users   Sequences    Positive
--------------------------------------------
Train            600      198005         790
Validation       200       65167         264
Test             200       65280         279
--------------------------------------------

Resampled training sequences: 1580 | Positives: 790
OFFICIAL USER SPLIT (from week1/week1_official_split.py)
Total users:       1000
Training users:    600 (60%)
Validation users:  200 (20%)
Testing users:     200 (20%)
Train ∩ Validation: 0
Train ∩ Test:       0
Validation ∩ Test:  0

Epoch 1/20 | Loss: 0.6805
Epoch 5/20 | Loss: 0.5814
Epoch 10/20 | Loss: 0.4942
Epoch 15/20 | Loss: 0.4435
Epoch 20/20 | Loss: 0.3992
--- LOGON AGENT EVALUATIO